In [3]:
!pip install gymnasium stable-baselines3 tensorboardX pandas --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 4.0 MB/s eta 0:00:00


In [4]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np


class AdaptiveDifficultyEnv(gym.Env):

    metadata = {"render_modes": ["human"]}

    def __init__(self, target_win_rate=0.55, max_steps=300):
        super().__init__()

        self.target_win_rate = target_win_rate
        self.max_steps = max_steps

        self.action_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(1,),
            dtype=np.float32
        )

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(5,),
            dtype=np.float32
        )

        self.step_count = 0
        self.difficulty = 0.5
        self.history = []
        self.streak = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.step_count = 0
        self.difficulty = 0.5
        self.history = []
        self.streak = 0

        obs = self._make_obs()

        return obs, {}

    def _simulate_player_outcome(self, difficulty):
        base_p = 0.8 - 0.6 * difficulty

        p_success = np.clip(
            base_p + self.np_random.normal(0, 0.1),
            0.05,
            0.95
        )

        return 1 if self.np_random.random() < p_success else 0

    def _make_obs(self):
        recent_win_rate = (
            np.mean(self.history[-10:])
            if self.history
            else 0.5
        )

        last_outcome = (
            self.history[-1]
            if self.history
            else 0
        )

        time_pressure = min(
            self.step_count / self.max_steps,
            1.0
        )

        return np.array(
            [
                self.difficulty,
                last_outcome,
                recent_win_rate,
                self.streak,
                time_pressure
            ],
            dtype=np.float32
        )

    def step(self, action):
        action = np.clip(action, 0.0, 1.0)

        previous_difficulty = float(self.difficulty)

        self.difficulty = float(
            0.8 * self.difficulty +
            0.2 * action[0]
        )

        outcome = self._simulate_player_outcome(
            self.difficulty
        )

        self.history.append(outcome)

        if outcome == 1:
            self.streak = (
                self.streak + 1
                if self.streak >= 0
                else 1
            )
        else:
            self.streak = (
                self.streak - 1
                if self.streak <= 0
                else -1
            )

        recent_win = float(
            np.mean(self.history[-10:])
        )

        reward_win = -abs(
            recent_win - self.target_win_rate
        )

        difficulty_change = abs(
            self.difficulty - previous_difficulty
        )

        reward_smooth = -0.1 * difficulty_change
        reward_engage = 0.01

        reward = float(
            reward_win +
            reward_smooth +
            reward_engage
        )

        self.step_count += 1

        terminated = (
            self.step_count >= self.max_steps
        )

        truncated = False

        obs = self._make_obs()

        info = {
            "difficulty": float(self.difficulty),
            "win": int(outcome),
            "recent_win_rate": recent_win,
            "difficulty_change": float(
                self.difficulty - previous_difficulty
            ),
        }

        return (
            obs,
            reward,
            terminated,
            truncated,
            info,
        )


env = AdaptiveDifficultyEnv()

obs, _ = env.reset()

print("Obs space:", env.observation_space)
print("Act space:", env.action_space)

for _ in range(10):
    action = env.action_space.sample()

    obs, rew, term, trunc, info = env.step(action)

    if term or trunc:
        break

print("Env OK, sample obs:", obs)
print("Final Reward:", rew)
print("Info:", info)

Obs space: Box(-inf, inf, (5,), float32)
Act space: Box(0.0, 1.0, (1,), float32)
Env OK, sample obs: [0.45466244 1.         0.5        1.         0.03333334]
Final Reward: -0.046329733133316085
Info: {'difficulty': 0.4546624422073364, 'win': 1, 'recent_win_rate': 0.5, 'difficulty_change': 0.0632973313331604}


In [5]:
import os
import numpy as np
import pandas as pd

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("logs_ppo", exist_ok=True)

print("Folders Ready!")

Folders Ready!


In [7]:
vec_env = make_vec_env(
    lambda: AdaptiveDifficultyEnv(
        target_win_rate=0.55,
        max_steps=300
    ),
    n_envs=4
)

print("Vectorized Environment Ready!")

Vectorized Environment Ready!


In [8]:
model = PPO(
    "MlpPolicy",
    vec_env,
    verbose=1,
    tensorboard_log="./logs_ppo",
    seed=42
)

print("PPO Model Created!")

Using cpu device
PPO Model Created!


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [9]:
model.learn(
    total_timesteps=50_000,
    progress_bar=True
)

Logging to ./logs_ppo/PPO_1


Output()

/usr/local/lib/python3.13/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 300      |
|    ep_rew_mean     | -42.4    |
| time/              |          |
|    fps             | 2985     |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 8192     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 300         |
|    ep_rew_mean          | -41.6       |
| time/                   |             |
|    fps                  | 1221        |
|    iterations           | 2           |
|    time_elapsed         | 13          |
|    total_timesteps      | 16384       |
| train/                  |             |
|    approx_kl            | 0.006833205 |
|    clip_fraction        | 0.0636      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.4        |
|    explained_variance   | 0.0306      |
|    learning_rate        | 0.

In [10]:
model.save("./models/ppo_dda")

print("Your Model is Saved")

Your Model is Saved


***Common Evaluation Function***

In [11]:
def evaluate_controller(env, action_function, n_episodes=50):

    episode_lengths = []
    recent_win_rates = []
    difficulty_change_variances = []

    for _ in range(n_episodes):

        obs, info = env.reset()

        terminated = False
        truncated = False

        difficulty_changes = []
        steps = 0

        while not (terminated or truncated):

            action = action_function(obs, info)

            obs, reward, terminated, truncated, info = env.step(action)

            difficulty_changes.append(
                info["difficulty_change"]
            )

            steps += 1

        episode_lengths.append(steps)

        recent_win_rates.append(
            info["recent_win_rate"]
        )

        difficulty_change_variances.append(
            np.var(difficulty_changes)
        )

    return {
        "mean_recent_win_rate": float(
            np.mean(recent_win_rates)
        ),
        "variance_difficulty_changes": float(
            np.mean(difficulty_change_variances)
        ),
        "average_episode_length": float(
            np.mean(episode_lengths)
        ),
    }

## ***Evaluate PPO***

In [12]:
ppo_eval_env = AdaptiveDifficultyEnv(
    target_win_rate=0.55,
    max_steps=300
)


def ppo_action(obs, info):

    action, _ = model.predict(
        obs,
        deterministic=True
    )

    return action


ppo_metrics = evaluate_controller(
    ppo_eval_env,
    ppo_action,
    n_episodes=50
)

print("PPO Metrics:")
print(ppo_metrics)

PPO Metrics:
{'mean_recent_win_rate': 0.602, 'variance_difficulty_changes': 0.0028543675335989927, 'average_episode_length': 300.0}


## ***Static Difficulty Baseline***

In [13]:
static_difficulties = [0.2, 0.5, 0.8]

static_results = {}

for difficulty in static_difficulties:

    env = AdaptiveDifficultyEnv(
        target_win_rate=0.55,
        max_steps=300
    )

    def static_action(obs, info, d=difficulty):
        return np.array(
            [d],
            dtype=np.float32
        )

    metrics = evaluate_controller(
        env,
        static_action,
        n_episodes=50
    )

    static_results[f"Static_{difficulty}"] = metrics


print("Static Baseline Results:")

for method, metrics in static_results.items():
    print(method, metrics)

Static Baseline Results:
Static_0.2 {'mean_recent_win_rate': 0.66, 'variance_difficulty_changes': 3.233333488318279e-05, 'average_episode_length': 300.0}
Static_0.5 {'mean_recent_win_rate': 0.48599999999999993, 'variance_difficulty_changes': 0.0, 'average_episode_length': 300.0}
Static_0.8 {'mean_recent_win_rate': 0.332, 'variance_difficulty_changes': 3.233335052052786e-05, 'average_episode_length': 300.0}


## ***Rule Based DDA Controller***

In [14]:
class RuleBasedDDA:

    def __init__(
        self,
        initial_difficulty=0.5,
        step_size=0.1
    ):
        self.initial_difficulty = initial_difficulty
        self.step_size = step_size

        self.reset()

    def reset(self):

        self.difficulty = self.initial_difficulty

        self.consecutive_wins = 0
        self.consecutive_losses = 0

    def get_action(self):

        return np.array(
            [self.difficulty],
            dtype=np.float32
        )

    def update(self, win):

        if win == 1:

            self.consecutive_wins += 1
            self.consecutive_losses = 0

        else:

            self.consecutive_losses += 1
            self.consecutive_wins = 0

        if self.consecutive_wins >= 2:

            self.difficulty = min(
                1.0,
                self.difficulty + self.step_size
            )

            self.consecutive_wins = 0

        elif self.consecutive_losses >= 2:

            self.difficulty = max(
                0.0,
                self.difficulty - self.step_size
            )

            self.consecutive_losses = 0

In [15]:
def evaluate_rule_based(
    env,
    controller,
    n_episodes=50
):

    episode_lengths = []
    recent_win_rates = []
    difficulty_change_variances = []

    for _ in range(n_episodes):

        obs, info = env.reset()

        controller.reset()

        terminated = False
        truncated = False

        difficulty_changes = []
        steps = 0

        while not (terminated or truncated):

            action = controller.get_action()

            obs, reward, terminated, truncated, info = env.step(action)

            controller.update(
                info["win"]
            )

            difficulty_changes.append(
                info["difficulty_change"]
            )

            steps += 1

        episode_lengths.append(steps)

        recent_win_rates.append(
            info["recent_win_rate"]
        )

        difficulty_change_variances.append(
            np.var(difficulty_changes)
        )

    return {
        "mean_recent_win_rate": float(
            np.mean(recent_win_rates)
        ),
        "variance_difficulty_changes": float(
            np.mean(difficulty_change_variances)
        ),
        "average_episode_length": float(
            np.mean(episode_lengths)
        ),
    }

In [16]:
rule_controller = RuleBasedDDA(
    initial_difficulty=0.5,
    step_size=0.1
)

rule_env = AdaptiveDifficultyEnv(
    target_win_rate=0.55,
    max_steps=300
)

rule_metrics = evaluate_rule_based(
    rule_env,
    rule_controller,
    n_episodes=50
)

print("Rule-Based DDA Metrics:")
print(rule_metrics)

Rule-Based DDA Metrics:
{'mean_recent_win_rate': 0.516, 'variance_difficulty_changes': 0.00035104306589388816, 'average_episode_length': 300.0}


## ***Combine Results***

In [17]:
results = [
    {
        "method": "PPO",
        **ppo_metrics
    }
]

for method, metrics in static_results.items():

    results.append(
        {
            "method": method,
            **metrics
        }
    )

results.append(
    {
        "method": "RuleBased_DDA",
        **rule_metrics
    }
)

results_df = pd.DataFrame(results)

results_df

,method,mean_recent_win_rate,variance_difficulty_changes,average_episode_length
0,PPO,0.602,0.002854,300.0
1,Static_0.2,0.660,0.000032,300.0
2,Static_0.5,0.486,0.000000,300.0
3,Static_0.8,0.332,0.000032,300.0
4,RuleBased_DDA,0.516,0.000351,300.0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## ***Save..!***

In [20]:
metrics_path = "./results/metrics.csv"

results_df.to_csv(
    metrics_path,
    index=False
)

print(f"Metrics saved to: {metrics_path}")

results_df

Metrics saved to: ./results/metrics.csv


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,method,mean_recent_win_rate,variance_difficulty_changes,average_episode_length
0,PPO,0.602,0.002854,300.0
1,Static_0.2,0.660,0.000032,300.0
2,Static_0.5,0.486,0.000000,300.0
3,Static_0.8,0.332,0.000032,300.0
4,RuleBased_DDA,0.516,0.000351,300.0


In [21]:
pd.read_csv("./results/metrics.csv")

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,method,mean_recent_win_rate,variance_difficulty_changes,average_episode_length
0,PPO,0.602,0.002854,300.0
1,Static_0.2,0.660,0.000032,300.0
2,Static_0.5,0.486,0.000000,300.0
3,Static_0.8,0.332,0.000032,300.0
4,RuleBased_DDA,0.516,0.000351,300.0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
